# Dot Density Maps

In [ ]:
%load_ext autoreload
%autoreload 2

import geodatasets
import geopandas as gpd
from tobler.util import dot_density
from tobler.dasymetric import masked_dot_density

Dot density maps are a useful pedagogical tool for understanding the uniform-distribution assumption made during areal interpolation exercises. They can also be very effective maps for visualizing relative frequency among multiple categories. To generate a dot-density map, we draw random points inside each polygon from an input geodataframe according to given data-generating process (DGP) (e.g. 'uniform' by default), with the size of the draw equal to the value specified by a (set of) column(s) in the dataframe.

For example, for a given extensive variable, like total_population, this function can be used to generate one stochastic layout of the hypothetical population distribution implied by areal interpolation (the uniform distribution in this case). Applying the function to several extensive variables, like subsets of the total population (e.g. age, race, or education) allows visualization of both the distributional assumptions, but also the relative segregation tendencies of the groups.

This notebook shows:

1. how the typical assumption of uniform distribution throughout `source` materializes in practice
2. how alternative DGPs could result in different hypothetical layouts
3. how dasymetric refinement changes the distribution assumptions by restricting `source` via a masking layer

In [ ]:
path = geodatasets.get_path("geoda tampa1")

gdf = gpd.read_file(path)
gdf.head()

In [ ]:
race_cols = ["WHITE_", "BLACK_", "ASIAN_", "HISP_"]
age_cols = ["POP_16", "POP_65"]

In [ ]:
gdf.explore()

## `dot_density`

The `scale` parameter creates *proportional* dot-density maps. For visualization rather than analysis, it's often more efficient (and sometimes more effective) to map proportional densities rather than a 1:1 representation.

### Uniform distribution

In [ ]:
race_points = dot_density(gdf, columns=race_cols, scale=0.4)

In [ ]:
race_points

the number of rows in the dataframe should equal the scaled sum of the input columns

In [ ]:
(gdf[race_cols] * 0.4).round(0).sum().sum()

when mapping, you need to shuffle the points, otherwise they draw one category at a time

In [ ]:
race_points.sample(frac=1).plot(
    "category",
    categorical=True,
    cmap="rainbow",
    figsize=(8, 8),
    markersize=0.05,
    alpha=0.2,
    legend=True,
).axis("off")

In [ ]:
age_points = dot_density(gdf, columns=age_cols, scale=0.5)

In [ ]:
age_points.sample(frac=1).plot(
    "category",
    categorical=True,
    figsize=(8, 8),
    markersize=0.05,
    alpha=0.2,
    cmap="rainbow",
    legend=True,
).axis("off")

### Alternative DGPs

Since the function is doing little more than [sampling points](https://geopandas.org/en/latest/docs/user_guide/sampling.html) by category, you can also consider different theoretical generating processes to distribute points within each polygon. If we think that population tends to cluster (by group), we might consider what clustered point processes look like

In [ ]:
age_points_cluster_normal = dot_density(
    gdf, columns=age_cols, scale=0.5, method="cluster_normal"
)

In [ ]:
age_points_cluster_normal.sample(frac=1).plot(
    "category",
    categorical=True,
    figsize=(8, 8),
    markersize=0.05,
    alpha=0.2,
    cmap="rainbow",
    legend=True,
).axis("off")

an interesting thing about the non-uniform DGPs is that they're applied separately to each category. At the tract level, the segregation measured between categories will remain consistent because it's still the same people sharing the same unit. At the sub-tract level (and most likely cross-tract, whatever you call irregularly overlaid alternative polygons) this will induce *greater* levels of segregation between categories because the seeds of the DGP are chosen independently, so the groups cluster around different loci within the polygon

In [ ]:
age_points_cluster_poisson = dot_density(
    gdf, columns=age_cols, scale=0.5, method="cluster_poisson"
)

In [ ]:
age_points_cluster_poisson.sample(frac=1).plot(
    "category",
    categorical=True,
    figsize=(8, 8),
    markersize=0.05,
    alpha=0.2,
    cmap="rainbow",
    legend=True,
).axis("off")

## Masked Dot Density

Dasymetric interpolation attempts to improve on the uniform distribution assumption by constraining the source polygons to the areas where the distributional assumptions are more likely to hold. For population studies, this generally means using additional data to mask out uninhabited areas such that `source` refers to areas where people actually live. Applying the uniform distribution assumption to *this* set of polygons is still error-prone, but is more realistic than the generic assumption without dasymetric refinement.

In [ ]:
raster = "s3://spatial-ucr/nlcd/landcover/nlcd_landcover_2021.tif"

In [ ]:
import rasterio

with rasterio.Env(AWS_NO_SIGN_REQUEST='YES'):
    masked_age_points = masked_dot_density(
        gdf, raster=raster, pixel_values=[22, 23, 24], columns=age_cols
    )

In [ ]:
masked_age_points.sample(frac=1).plot(
    "category",
    categorical=True,
    figsize=(8, 8),
    markersize=0.05,
    alpha=0.2,
    cmap="rainbow",
    legend=True,
).axis("off")

In [ ]:
with rasterio.Env(AWS_NO_SIGN_REQUEST="YES"):
    masked_age_points_normal = masked_dot_density(
        gdf, raster=raster, pixel_values=[22, 23, 24], columns=age_cols, method="normal"
    )

In [ ]:
masked_age_points_normal.sample(frac=1).plot(
    "category",
    categorical=True,
    figsize=(8, 8),
    markersize=0.05,
    alpha=0.2,
    cmap="rainbow",
    legend=True,
).axis("off")

In [ ]:
with rasterio.Env(AWS_NO_SIGN_REQUEST="YES"):
    masked_age_points_cluster = masked_dot_density(
        gdf,
        raster=raster,
        pixel_values=[22, 23, 24],
        columns=age_cols,
        method="cluster_normal",
    )

In [ ]:
masked_age_points_cluster.sample(frac=1).plot(
    "category",
    categorical=True,
    figsize=(8, 8),
    markersize=0.05,
    alpha=0.2,
    cmap="rainbow",
    legend=True,
).axis("off")

## Interactive viz

there are too many points to use the geopandas `explore` method, but `lonboard` can handle these maps with ease

In [ ]:
import numpy as np
from lonboard import viz
from lonboard.colormap import DiscreteColormap, apply_categorical_cmap
from matplotlib import colormaps

In [ ]:
c = [colormaps.get_cmap("rainbow")(i, bytes=True) for i in np.linspace(0, 1, 4)]

In [ ]:
simple_cmap = dict(zip(race_cols, c))

In [ ]:
race_points = race_points.sample(frac=1)

In [ ]:
fill_color = apply_categorical_cmap(race_points["category"], simple_cmap)

In [ ]:
viz(race_points, scatterplot_kwargs=dict(get_fill_color=fill_color))

In [ ]:
c = [colormaps.get_cmap("rainbow")(i, bytes=True) for i in np.linspace(0, 1, 2)]
simple_cmap = dict(zip(age_cols, c))

In [ ]:
age_points = age_points.sample(frac=1)
fill_color = apply_categorical_cmap(age_points["category"], simple_cmap)
viz(age_points, scatterplot_kwargs=dict(get_fill_color=fill_color))

In [ ]:
age_points_cluster_normal = age_points_cluster_normal.sample(frac=1)
fill_color = apply_categorical_cmap(age_points_cluster_normal["category"], simple_cmap)
viz(age_points_cluster_normal, scatterplot_kwargs=dict(get_fill_color=fill_color))

In [ ]:
age_points_cluster_poisson = age_points_cluster_poisson.sample(frac=1)
fill_color = apply_categorical_cmap(age_points_cluster_poisson["category"], simple_cmap)
viz(age_points_cluster_poisson, scatterplot_kwargs=dict(get_fill_color=fill_color))

In [ ]:
masked_age_points = masked_age_points.sample(frac=1)
fill_color = apply_categorical_cmap(masked_age_points["category"], simple_cmap)
viz(masked_age_points, scatterplot_kwargs=dict(get_fill_color=fill_color))

In [ ]:
masked_age_points_normal = masked_age_points_cluster.sample(frac=1)
fill_color = apply_categorical_cmap(masked_age_points_normal["category"], simple_cmap)
viz(masked_age_points_normal, scatterplot_kwargs=dict(get_fill_color=fill_color))